# Fase 4 - Parte A: grafo de vecindad 

SKATER solo puede poner en una misma región eventos que estén conectados en un grafo de vecindad. Esta parte construye ese grafo, es decir, decide qué coronas son vecinas, y lo guarda para las partes B y C, que corren en R.

El grafo se construye con tres reglas 

1. La triangulación de Delaunay une cada corona con sus vecinos naturales, las coronas con las que comparte un triángulo.
2. Solo se conservan las aristas de hasta 1000 m, valor heredado del alcance de los variogramas de marcas de la Fase 2. Se considera que hasta esa distancia los eventos vecinos se parecen.
3.  El umbral deja muchos grupos sueltos (cúmulos). Se unen con las aristas más cortas de la triangulación que conectan grupos todavía separados, hasta que todo queda en un solo grafo (algoritmo de Kruskal).


**Datos de entrada**

Usa las coronas de la Fase 3 con las anomalías y los eventos bajo el umbrla de la fase 0. 
La capa con las siete variables, las categóricas (GRUPO, TIPO, GEOLOGIA, GEOMORFOLOGIA, COBERTURA) y EN_MODELO.
y usa el eje del río

**Salidas**

|capa|Contenido|
|---|---|
|eventos_f4|Un punto por evento con su identificador, la componente del grafo, el cúmulo, la ladera, las siete variables y los campos para caracterizar las regiones.|
|grafo_f4_aristas|Todas las aristas de Delaunay como líneas, con su largo, su distancia al río, si lo cruzan y su estado: conservada, puente, larga (descartada) o coincidente.|
|grafo_f4_tabla|El grafo final como tabla de vecinos con las dos direcciones de cada arista|

Aparte, también se guardan los parametros: el umbral y los conteos del grafo.



In [13]:
import os # rutas de salida 
import arcpy # Lectura y creación de capas de ArcGIS Pro

# Manejo de coordenadas y tablas
import numpy as np
import pandas as pd

# Triangulación de las coronas 
from scipy.spatial import Delaunay, cKDTree

# matriz de conexiones del grafo y conteo de grupos conectados.
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
 

## Configuración

### **¿Qué hace?**

Reúne las rutas, los campos y los umbrales que se van a probar. Es el bloque editable 



In [14]:
# Rutas y archivos
BASE = r"D:\Users\tasal\Desktop\Tesis de grado"
CARPETA = BASE + r"\fase4"
GDB = CARPETA + r"\fase4.gdb"
SWM = CARPETA + r"\grafo_f4.swm"


# Coronas de la fase 3 e identificador 
CORONAS = r"D:\Users\tasal\Desktop\Tesis de grado\fase3\F3_fase3_integrado.gpkg\main.fase3_eventos"
CAMPO_ID = "ID"

# Tabla de variables de la fase 3 y su identificador
TABLA_VARIABLES = r"D:/Users/tasal/Desktop/Tesis de grado/fase3/F3_S2_tabla_modelo.csv"   # copiar del diagnostico
CAMPO_ID_TABLA = "ID"


# as variables con que SKATER arma las regiones: log₁₀ del área, del H/L y de la elongación, pendiente,
#log₁₀ de la distancia a estructuras y a drenajes, y densidad de lineamientos.

VARIABLES = ["LOG_AREA", "LOG_HL", "LOG_ELONG",
             "PENDIENTE", "LOG_DIST_ESTRUCTURA", "LOG_DIST_DRENAJE", "DENS_LIN_1000"]



# Campos que no entran a SKATER, pero se copian a eventos_f4 para caracterizar las regiones en la Parte D.

CARACT_TABLA = ["GRUPO", "TIPO", "GEOLOGIA", "GEOMORFOLOGIA", "COBERTURA"]   # para la Parte D

CARACT_CAPA = ["ANOMALIA", "BAJO_UMBRAL"]  # para la Parte D

# Capa del eje del río. 
RIO = "Rio_Medellin"


REVISAR_RIO_M = 100        # aristas conservadas a menos de esta distancia del rio se listan
UMBRAL_M = 1000            # vecindad: alcance de los variogramas de marcas de area y H/L (Fase 2)
RESPETAR_RIO = False       # False = un solo grafo; el rio no interviene en su construccion 
                           # True  = version anterior: ninguna arista cruza el rio (dos grafos)
TAM_MIN_REGION = 20        # solo para reportar cumulos pequenos
TOL_COINCIDENCIA = 2.0     # m: coronas a menos de esta distancia se reportan como coincidentes
PARAMETROS = CARPETA + r"\F4_A_parametros.csv"   # la Parte B (R) lee de aqui el umbral

# Funciones de lectura y decisiones

### **Qué hace**

Prepara la lectura de los datos y deja escritas en el reporte las decisiones del grafo.


In [15]:
# =====================================================================================
 
arcpy.env.overwriteOutput = True
CAMPO_ID_TABLA = CAMPO_ID_TABLA.replace("\ufeff", "").strip()   # por si se copio con la marca BOM
 
 
def clave(v):
    """ID como texto comparable (evita que 73, 73.0 y '73' no coincidan)."""
    try:
        return str(int(float(v)))
    except (TypeError, ValueError):
        return str(v).strip()
 
 
def leer_tabla(ruta):
    if ruta.lower().endswith(".csv"):
        t = None
        for cod in ("utf-8-sig", "latin-1"):
            try:
                t = pd.read_csv(ruta, sep=None, engine="python", encoding=cod)
                break
            except UnicodeDecodeError:
                continue
        if t is None:
            raise SystemExit("No se pudo leer el CSV (codificacion).")
    else:
        campos_t = [f.name for f in arcpy.ListFields(ruta) if f.type not in ("Geometry", "Blob", "Raster")]
        t = pd.DataFrame(list(arcpy.da.SearchCursor(ruta, campos_t)), columns=campos_t)
    t.columns = [str(c).replace("\ufeff", "").strip() for c in t.columns]
    return t
 
 
print("=" * 70)
print("FASE 4 - PARTE A: GRAFO DE VECINDAD")
print("=" * 70)
print("Decisiones registradas:")
print("  1. Unidad: las coronas de la muestra del modelo de la Fase 3.")
print("  2. Variables: 4 condicionantes continuos de la Fase 3 y log10 de area, H/L y elongacion,")
print("     con las mismas transformaciones; sin pesos (ArcGIS estandariza cada variable).")
print("  3. Vecindad: triangulacion de Delaunay de las coronas (vecinos naturales).")
if RESPETAR_RIO:
    print("  4. Se eliminan las aristas que cruzan el eje del rio Medellin.")
else:
    print("  4. El rio NO interviene en el grafo: un solo grafo para todo el valle, para que SKATER")
    print("     decida si las laderas se separan. El eje del rio solo se usa para reportar.")
print(f"  5. Umbral de vecindad: {UMBRAL_M} m, alcance de los variogramas de marcas de area y H/L")
print("     (Fase 2). Se conservan las aristas de la triangulacion de hasta ese largo.")
print("  6. Puentes: los cumulos que quedan sueltos se reconectan con la arista mas corta de la")
print("     triangulacion entre ellos hasta que el grafo quede conectado.")
print("  7. Las coronas coincidentes se conservan; si alguna queda fuera de la triangulacion")
print("     (misma coordenada), se une a su corona mas cercana.")

FASE 4 - PARTE A: GRAFO DE VECINDAD
Decisiones registradas:
  1. Unidad: las coronas de la muestra del modelo de la Fase 3.
  2. Variables: 4 condicionantes continuos de la Fase 3 y log10 de area, H/L y elongacion,
     con las mismas transformaciones; sin pesos (ArcGIS estandariza cada variable).
  3. Vecindad: triangulacion de Delaunay de las coronas (vecinos naturales).
  4. El rio NO interviene en el grafo: un solo grafo para todo el valle, para que SKATER
     decida si las laderas se separan. El eje del rio solo se usa para reportar.
  5. Umbral de vecindad: 1000 m, alcance de los variogramas de marcas de area y H/L
     (Fase 2). Se conservan las aristas de la triangulacion de hasta ese largo.
  6. Puentes: los cumulos que quedan sueltos se reconectan con la arista mas corta de la
     triangulacion entre ellos hasta que el grafo quede conectado.
  7. Las coronas coincidentes se conservan; si alguna queda fuera de la triangulacion
     (misma coordenada), se une a su corona mas 

## Datos

### **¿Qué hace?**

Arma, para cada corona, su posición, sus siete variables y los campos de caracterización, y comprueba que la capa y la tabla sean la misma muestra de la Fase 3.

Además, lista los pares de coronas a menos de 2 m.

In [16]:
# ------------------------------------------------------------------ datos
desc = arcpy.Describe(CORONAS)
if not hasattr(desc, "spatialReference"):
    raise SystemExit(f"CORONAS no es una capa de puntos ({desc.dataType}): {CORONAS}")
sr = desc.spatialReference
campos_capa = [f.name for f in arcpy.ListFields(CORONAS)]
extra_capa = [c for c in CARACT_CAPA if c in campos_capa]
filas = list(arcpy.da.SearchCursor(CORONAS, [CAMPO_ID, "SHAPE@XY"] + extra_capa))
claves = [clave(f[0]) for f in filas]
xy = np.array([f[1] for f in filas], dtype=float)
vals_capa = {clave(f[0]): f[2:] for f in filas}
n = len(claves)
 
tabla = leer_tabla(TABLA_VARIABLES)
faltan = [c for c in [CAMPO_ID_TABLA] + VARIABLES if c not in tabla.columns]
if faltan:
    raise SystemExit(f"Faltan campos en la tabla: {faltan}\nCampos de la tabla: {list(tabla.columns)}")
for c in VARIABLES:
    if not pd.api.types.is_numeric_dtype(tabla[c]):
        tabla[c] = pd.to_numeric(tabla[c].astype(str).str.replace(",", "."), errors="coerce")
tabla.index = [clave(v) for v in tabla[CAMPO_ID_TABLA]]
tabla = tabla[~tabla.index.duplicated()].reindex(claves)
if tabla[VARIABLES].isna().any().any():
    raise SystemExit("Hay eventos sin variables: revisa la tabla con el diagnostico.")
extra_tabla = [c for c in CARACT_TABLA if c in tabla.columns]
print(f"\nDATOS\n  Eventos: {n} | variables: {len(VARIABLES)} | "
      f"campos de caracterizacion: {extra_tabla + extra_capa}")
tabla_completa = leer_tabla(TABLA_VARIABLES)
if "EN_MODELO" in tabla_completa.columns:
    en_modelo = tabla_completa["EN_MODELO"].astype(str).str.strip().str.upper().isin(["1", "1.0", "TRUE", "VERDADERO", "SI", "S\u00cd"])
    ids_modelo = {clave(v) for v in tabla_completa.loc[en_modelo, CAMPO_ID_TABLA]}
    fuera_modelo = [k for k in claves if k not in ids_modelo]
    print(f"  Control con la Fase 3: eventos con EN_MODELO = 1 en la tabla: {len(ids_modelo)} | "
          f"en la capa: {n} | de la capa fuera del modelo: {len(fuera_modelo)}")
    if len(ids_modelo) != n or fuera_modelo:
        print("  ATENCION: la capa y la tabla no corresponden a la misma muestra. Revisa que ambas")
        print("  rutas apunten a la corrida actual de la Fase 3.")
pares_coinc = sorted(cKDTree(xy).query_pairs(TOL_COINCIDENCIA))
print(f"  Coronas a menos de {TOL_COINCIDENCIA:.0f} m: {len(pares_coinc)} pares" +
      (": " + ", ".join(f"{claves[a]}-{claves[b]} ({np.linalg.norm(xy[a] - xy[b]):.1f} m)"
                        for a, b in pares_coinc) if pares_coinc else ""))


DATOS
  Eventos: 427 | variables: 7 | campos de caracterizacion: ['GRUPO', 'TIPO', 'GEOLOGIA', 'GEOMORFOLOGIA', 'COBERTURA', 'ANOMALIA', 'BAJO_UMBRAL']
  Control con la Fase 3: eventos con EN_MODELO = 1 en la tabla: 427 | en la capa: 427 | de la capa fuera del modelo: 0
  Coronas a menos de 2 m: 1 pares: 405-446 (2.0 m)


## Grafo

### **¿Qué hace?**

Construye el grafo con las tres reglas (Delaunay, umbral de 1 km y puentes) y calcula, solo para el reporte, qué aristas cruzan el río.

Esto se hace en el siguiente orden: 

1. Cada posición se triangula una vez  Si una corona queda por fuera, se une a su gemela con una arista coincidente, que siempre se conserva.

2. Cada triángulo aporta tres aristas; el conjunto ar evita contarlas dos veces. Después se mide el largo de cada una.

3. Lee los tramos del eje, los orienta de sur a norte y cierra los huecos de digitalización entre tramos con una franja angosta alrededor de la línea (la mitad del hueco mayor más 5 m, entre 10 y 100 m). Una arista cruza el río si toca la línea, o si atraviesa la franja sin que ninguno de sus extremos esté dentro de ella. También mide la distancia de cada arista al río.

4. Cuenta los grupos conectados con las aristas de hasta 1 km

5.  Recorre las aristas largas de la más corta a la más larga y agrega una solo si une dos grupos que todavía están separados.

6. Cada arista queda como conservada, puente, larga o coincidente. Imprime los conteos, el largo de los puentes, los diez más largos, las componentes del grafo final, las aristas que cruzan el río y el número de vecinos por evento.


In [17]:

_, primeros = np.unique(xy, axis=0, return_index=True)
primeros = np.sort(primeros)
tri = Delaunay(xy[primeros])
simplices = primeros[tri.simplices]
ar = set()
for s in simplices:
    for a, b in ((s[0], s[1]), (s[1], s[2]), (s[0], s[2])):
        ar.add((min(int(a), int(b)), max(int(a), int(b))))
        
# Coronas que la triangulacion deja fuera (misma coordenada que otra): se unen a la corona
# triangulada mas cercana (su gemela)
dentro_tri = sorted({int(v) for v in simplices.ravel()})
fuera_tri = sorted(set(range(n)) - set(dentro_tri))
if fuera_tri:
    _, pos = cKDTree(xy[dentro_tri]).query(xy[fuera_tri], k=1)
    for a, p_ in zip(fuera_tri, np.atleast_1d(pos)):
        b = dentro_tri[int(p_)]
        ar.add((min(a, b), max(a, b)))
ar = np.array(sorted(ar))
largo = np.linalg.norm(xy[ar[:, 0]] - xy[ar[:, 1]], axis=1)
forzada = np.array([(a in fuera_tri) or (b in fuera_tri) for a, b in ar])
 
# Eje del rio: se leen todos los tramos, se orientan de sur a norte (sentido del flujo)
tramos = []
with arcpy.da.SearchCursor(RIO, ["SHAPE@"]) as cur:
    for (g,) in cur:
        if g is None:
            continue
        if g.spatialReference.name != sr.name:
            g = g.projectAs(sr)
        for i in range(g.partCount):
            pts = [(p.X, p.Y) for p in g.getPart(i) if p]
            if len(pts) >= 2:
                if pts[-1][1] < pts[0][1]:
                    pts = pts[::-1]
                tramos.append(pts)
rio = arcpy.Polyline(arcpy.Array([arcpy.Array([arcpy.Point(x, y) for x, y in t]) for t in tramos]), sr)
 
k_tr = len(tramos)
extremos = np.array([t[0] for t in tramos] + [t[-1] for t in tramos], dtype=float)
duenio = np.array(list(range(k_tr)) * 2)
arbol_ext = cKDTree(extremos)
dd, jj = arbol_ext.query(extremos, k=min(6, len(extremos)))
hueco = np.array([next((d for d, j in zip(dd[i], jj[i]) if duenio[j] != duenio[i]), np.inf)
                  for i in range(len(extremos))])
huecos = np.sort(hueco)[:-2] if k_tr > 1 else np.array([0.0])   # los 2 mayores son nacimiento y salida
hueco_max = float(huecos.max()) if len(huecos) else 0.0
ANCHO_FRANJA = min(max(10.0, hueco_max / 2 + 5), 100.0)
franja = rio.buffer(ANCHO_FRANJA)
en_franja = np.array([arcpy.PointGeometry(arcpy.Point(*p), sr).distanceTo(rio) < ANCHO_FRANJA for p in xy])
 
lineas = [arcpy.Polyline(arcpy.Array([arcpy.Point(*xy[a]), arcpy.Point(*xy[b])]), sr) for a, b in ar]
# Cruza si toca la linea, o si pasa por la franja sin que ninguno de sus extremos este dentro de ella
cruza = np.array([(not l.disjoint(rio)) or
                  ((not l.disjoint(franja)) and not en_franja[a] and not en_franja[b])
                  for l, (a, b) in zip(lineas, ar)])
dist_rio = np.array([l.distanceTo(rio) for l in lineas])
 
 
def componentes(mascara):
    e = ar[mascara]
    A = coo_matrix((np.ones(len(e)), (e[:, 0], e[:, 1])), shape=(n, n))
    return connected_components(A, directed=False)
 
 
sin_rio = ~cruza if RESPETAR_RIO else np.ones(len(ar), dtype=bool)
corta = (sin_rio & (largo <= UMBRAL_M)) | forzada
 
# Cumulos: grupos de eventos unidos solo por aristas cortas
n_cum, cumulo = componentes(corta)
tam_cum = np.bincount(cumulo)
 
# Puentes: se recorren las aristas largas permitidas, de la mas corta a la mas larga.
padre = list(range(n_cum))
 
 
def raiz(x):
    while padre[x] != x:
        padre[x] = padre[padre[x]]
        x = padre[x]
    return x
 
 
puente = np.zeros(len(ar), dtype=bool)
candidatas = np.where(sin_rio & ~corta)[0]
for i in candidatas[np.argsort(largo[candidatas])]:
    ra, rb = raiz(cumulo[ar[i, 0]]), raiz(cumulo[ar[i, 1]])
    if ra != rb:
        padre[ra] = rb
        puente[i] = True
 
conservar = corta | puente
estado = np.where(corta, "conservada", np.where(puente, "puente", "larga"))
estado = np.where(forzada, "coincidente", estado)
if RESPETAR_RIO:
    estado = np.where(cruza, "cruza_rio", estado)
ncomp, comp = componentes(conservar)
grado = np.bincount(ar[conservar].ravel(), minlength=n)
lp = largo[puente]
peq = tam_cum < TAM_MIN_REGION
 
print("\nGRAFO")
print(f"  Aristas de Delaunay: {len(ar) - int(forzada.sum())}"
      + (f" | coronas fuera de la triangulacion unidas a su vecina: {len(fuera_tri)}" if fuera_tri else ""))
if RESPETAR_RIO:
    print(f"  Eliminadas por cruzar el rio: {int(cruza.sum())}")
print(f"  Aristas cortas (<= {UMBRAL_M} m): {int(corta.sum())}")
print(f"  Cumulos que dejan las aristas cortas: {n_cum} | de 1 solo evento: {int((tam_cum == 1).sum())} | "
      f"de menos de {TAM_MIN_REGION} eventos: {int(peq.sum())} (suman {int(tam_cum[peq].sum())} eventos)")
print(f"  Cumulos mas grandes: {np.sort(tam_cum)[::-1][:6].tolist()}")
print(f"  ARISTAS PUENTE agregadas: {int(puente.sum())}")
if puente.any():
    print(f"    Largo de los puentes (m): minimo {lp.min():.0f} | mediana {np.median(lp):.0f} | maximo {lp.max():.0f}")
    print("    Los 10 puentes mas largos:")
    for i in candidatas[puente[candidatas]][np.argsort(largo[candidatas[puente[candidatas]]])][::-1][:10]:
        a, b = ar[i]
        print(f"      {claves[a]} - {claves[b]} | {largo[i]:.0f} m | une cumulos de "
              f"{tam_cum[cumulo[a]]} y {tam_cum[cumulo[b]]} eventos")
print(f"  Aristas largas descartadas (ni cortas ni puente): {int((estado == 'larga').sum())}")
print(f"  Grafo final: {int(conservar.sum())} aristas | componentes: {ncomp}")
if not RESPETAR_RIO:
    print(f"  Aristas del grafo final que cruzan el rio: cortas {int((corta & cruza).sum())} | "
          f"puentes {int((puente & cruza).sum())}")
print(f"  Vecinos por evento: min {grado.min()} | mediana {np.median(grado):.0f} | max {grado.max()} | "
      f"con un solo vecino: {int((grado == 1).sum())}")


GRAFO
  Aristas de Delaunay: 1266
  Aristas cortas (<= 1000 m): 664
  Cumulos que dejan las aristas cortas: 99 | de 1 solo evento: 41 | de menos de 20 eventos: 93 (suman 280 eventos)
  Cumulos mas grandes: [27, 26, 26, 26, 22, 20]
  ARISTAS PUENTE agregadas: 98
    Largo de los puentes (m): minimo 1001 | mediana 1523 | maximo 6381
    Los 10 puentes mas largos:
      254 - 316 | 6381 m | une cumulos de 1 y 1 eventos
      253 - 256 | 6221 m | une cumulos de 1 y 1 eventos
      3 - 281 | 6215 m | une cumulos de 1 y 2 eventos
      40 - 49 | 5357 m | une cumulos de 1 y 2 eventos
      30 - 40 | 5227 m | une cumulos de 1 y 1 eventos
      130 - 326 | 5210 m | une cumulos de 1 y 1 eventos
      270 - 326 | 4884 m | une cumulos de 1 y 1 eventos
      357 - 365 | 4250 m | une cumulos de 1 y 1 eventos
      30 - 326 | 4106 m | une cumulos de 1 y 1 eventos
      26 - 70 | 4031 m | une cumulos de 5 y 2 eventos
  Aristas largas descartadas (ni cortas ni puente): 504
  Grafo final: 762 aristas |

## Revisión del río

### **¿Qué hace?***

Describe cómo se relaciona el grafo con el río, sin cambiarlo: si las regiones podían pasar de una ladera a otra y por cuántas aristas.


In [18]:
# ------------------------------------------------------------------ revision del rio
print("\nREVISION DEL RIO")
print(f"  Tramos de la linea del rio: {k_tr} (orientados de sur a norte)")
print(f"  Hueco mayor entre tramos: {hueco_max:.1f} m | franja para cerrar huecos: {ANCHO_FRANJA:.0f} m a cada lado")
if hueco_max > 2 * 100.0:
    print("  ATENCION: hay huecos mayores que la franja maxima; revisar la linea del rio en el mapa.")
print(f"  Huecos mayores que 1 m: {int((huecos > 1).sum())} | eventos dentro de la franja: {int(en_franja.sum())}")
lado = np.array([rio.queryPointAndDistance(arcpy.PointGeometry(arcpy.Point(*p), sr))[3] for p in xy])
print(f"  Eventos a cada lado de la linea: {int(lado.sum())} | {int((~lado).sum())}")
entre = np.where(conservar & (lado[ar[:, 0]] != lado[ar[:, 1]]))[0]
etiq_entre = ("Aristas del grafo que unen las dos laderas" if not RESPETAR_RIO
              else "Aristas conservadas que unen los dos lados sin cruzar la linea")
print(f"  {etiq_entre}: {len(entre)} (cortas {int(corta[entre].sum())} | puentes {int(puente[entre].sum())})")
for i in entre[np.argsort(largo[entre])][::-1][:15]:
    print(f"    {claves[ar[i, 0]]} - {claves[ar[i, 1]]} | largo {largo[i]:.0f} m | "
          f"{'puente' if puente[i] else 'corta'} | a {dist_rio[i]:.0f} m del rio")
for c in range(ncomp):
    m = comp == c
    n_p = int((puente & (comp[ar[:, 0]] == c)).sum())
    n_cu = len(set(cumulo[m].tolist()))
    print(f"  Componente {c + 1}: {int(m.sum())} eventos | lado 1: {int(lado[m].sum())} | "
          f"lado 2: {int((~lado[m]).sum())} | cumulos: {n_cu} | puentes: {n_p}")
cerca = np.where(conservar & (dist_rio < REVISAR_RIO_M))[0]
print(f"  Aristas conservadas a menos de {REVISAR_RIO_M} m del rio: {len(cerca)}")
 


REVISION DEL RIO
  Tramos de la linea del rio: 30 (orientados de sur a norte)
  Hueco mayor entre tramos: 653.2 m | franja para cerrar huecos: 100 m a cada lado
  ATENCION: hay huecos mayores que la franja maxima; revisar la linea del rio en el mapa.
  Huecos mayores que 1 m: 58 | eventos dentro de la franja: 1
  Eventos a cada lado de la linea: 90 | 337
  Aristas del grafo que unen las dos laderas: 8 (cortas 3 | puentes 5)
    253 - 256 | largo 6221 m | puente | a 0 m del rio
    30 - 326 | largo 4106 m | puente | a 0 m del rio
    23 - 52 | largo 3619 m | puente | a 0 m del rio
    20 - 313 | largo 2958 m | puente | a 0 m del rio
    359 - 395 | largo 1297 m | puente | a 0 m del rio
    192 - 196 | largo 758 m | corta | a 0 m del rio
    192 - 195 | largo 667 m | corta | a 0 m del rio
    196 - 198 | largo 387 m | corta | a 0 m del rio
  Componente 1: 427 eventos | lado 1: 90 | lado 2: 337 | cumulos: 99 | puentes: 98
  Aristas conservadas a menos de 100 m del rio: 9


# Salidas

### **Qué hace**

Guarda los eventos y el grafo en los formatos que necesitan R y ArcGIS.



In [19]:

print("\nSALIDAS")
os.makedirs(CARPETA, exist_ok=True)          # crea la carpeta de salida si no existe
if not arcpy.Exists(GDB):
    arcpy.management.CreateFileGDB(CARPETA, os.path.basename(GDB))
 
fc_ev = os.path.join(GDB, "eventos_f4")
arcpy.management.CreateFeatureclass(GDB, "eventos_f4", "POINT", spatial_reference=sr)
arcpy.management.AddField(fc_ev, "ID", "LONG")
arcpy.management.AddField(fc_ev, "COMPONENTE", "SHORT")
arcpy.management.AddField(fc_ev, "CUMULO", "LONG")
arcpy.management.AddField(fc_ev, "LADERA", "SHORT")
for c in VARIABLES:
    arcpy.management.AddField(fc_ev, c, "DOUBLE")
for c in extra_tabla + extra_capa:
    arcpy.management.AddField(fc_ev, c, "TEXT", field_length=150)
with arcpy.da.InsertCursor(fc_ev, ["SHAPE@XY", "ID", "COMPONENTE", "CUMULO", "LADERA"] + VARIABLES + extra_tabla + extra_capa) as cur:
    for j, (k, p) in enumerate(zip(claves, xy)):
        fila = tabla.loc[k]
        vt = [None if pd.isna(fila[c]) else str(fila[c]) for c in extra_tabla]
        vc = [None if v is None else str(v) for v in vals_capa[k]]
        cur.insertRow([tuple(p), int(k), int(comp[j]) + 1, int(cumulo[j]) + 1, 1 if lado[j] else 2]
                      + [float(fila[c]) for c in VARIABLES] + vt + vc)
print(f"  {fc_ev}")
 
fc_ar = os.path.join(GDB, "grafo_f4_aristas")
arcpy.management.CreateFeatureclass(GDB, "grafo_f4_aristas", "POLYLINE", spatial_reference=sr)
for nombre, tipo in (("ID_A", "LONG"), ("ID_B", "LONG"), ("LARGO_M", "DOUBLE"),
                     ("DIST_RIO_M", "DOUBLE"), ("CRUZA_RIO", "SHORT"), ("ESTADO", "TEXT")):
    arcpy.management.AddField(fc_ar, nombre, tipo)
with arcpy.da.InsertCursor(fc_ar, ["SHAPE@", "ID_A", "ID_B", "LARGO_M", "DIST_RIO_M", "CRUZA_RIO", "ESTADO"]) as cur:
    for i, (a, b) in enumerate(ar):
        cur.insertRow([lineas[i], int(claves[a]), int(claves[b]),
                       float(largo[i]), float(dist_rio[i]), int(cruza[i]), str(estado[i])])
print(f"  {fc_ar}")
 
tb = os.path.join(GDB, "grafo_f4_tabla")
arcpy.management.CreateTable(GDB, "grafo_f4_tabla")
for nombre, tipo in (("ID", "LONG"), ("NID", "LONG"), ("WEIGHT", "DOUBLE")):
    arcpy.management.AddField(tb, nombre, tipo)
# ArcGIS exige que todos los vecinos de un mismo evento esten en filas consecutivas:
# se escriben ambas direcciones de cada arista y se ordenan por ID y luego por NID.
pares = sorted({(int(claves[a]), int(claves[b])) for a, b in ar[conservar]} |
               {(int(claves[b]), int(claves[a])) for a, b in ar[conservar]})
with arcpy.da.InsertCursor(tb, ["ID", "NID", "WEIGHT"]) as cur:
    for i, j in pares:
        cur.insertRow([i, j, 1.0])
print(f"  {tb}")
 
pd.DataFrame({"parametro": ["UMBRAL_M", "RESPETAR_RIO", "EVENTOS", "ARISTAS", "PUENTES", "CUMULOS"],
              "valor": [UMBRAL_M, int(RESPETAR_RIO), n, int(conservar.sum()), int(puente.sum()), n_cum]}
             ).to_csv(PARAMETROS, index=False)
print(f"  {PARAMETROS}")
 
# El .swm solo lo usa la comparacion opcional con ArcGIS (Parte B1); la Parte B en R lee grafo_f4_tabla.
try:
    res = arcpy.stats.GenerateSpatialWeightsMatrix(fc_ev, "ID", SWM, "CONVERT_TABLE",
                                                   "#", "#", "#", "#", "NO_STANDARDIZATION", tb)
    print(f"  {SWM}")
    print("\nMENSAJES DE ARCGIS AL CREAR EL .SWM")
    print(res.getMessages())
    tv = os.path.join(GDB, "grafo_f4_verificacion")
    arcpy.stats.ConvertSpatialWeightsMatrixtoTable(SWM, tv)
    n_swm = int(arcpy.management.GetCount(tv)[0])
    esperado = 2 * int(conservar.sum())
    print(f"\nVERIFICACION: el .swm tiene {n_swm} relaciones; se esperaban {esperado} "
          f"({'coinciden' if n_swm == esperado else 'NO coinciden: revisar'}).")
except arcpy.ExecuteError:
    print("\nAVISO: no se pudo crear el .swm (si dice que esta bloqueado, cierra y abre ArcGIS Pro).")
    print("La Parte B en R no lo necesita: usa grafo_f4_tabla. Mensajes de ArcGIS:")
    print(arcpy.GetMessages(2))
print("=" * 70)


SALIDAS
  D:\Users\tasal\Desktop\Tesis de grado\fase4\fase4.gdb\eventos_f4
  D:\Users\tasal\Desktop\Tesis de grado\fase4\fase4.gdb\grafo_f4_aristas
  D:\Users\tasal\Desktop\Tesis de grado\fase4\fase4.gdb\grafo_f4_tabla
  D:\Users\tasal\Desktop\Tesis de grado\fase4\F4_A_parametros.csv
  D:\Users\tasal\Desktop\Tesis de grado\fase4\grafo_f4.swm

MENSAJES DE ARCGIS AL CREAR EL .SWM
Start Time: sábado, 10 de octubre de 2026 3:47:39 a. m.
json:
{"element": "content", "data": "Constructing spatial weights based on input table...."}
json:
[{"element": "table", "data": [["Number of Features", "427"], ["Percentage of Spatial Connectivity", "0,84"], ["Average Number of Neighbors", "3,57"], ["Minimum Number of Neighbors", "1"], ["Maximum Number of Neighbors", "9"]], "elementProps": {"striped": "true", "noHeader": true, "title": "Spatial Weights Matrix Summary", "0": {"align": "left", "pad": "0px", "wrap": true}, "1": {"align": "left", "pad": "0px", "wrap": true}}}]
Succeeded at sábado, 10 de octu